In [1]:
import os
os.chdir('../../')

In [2]:
import os

# Obtém o diretório de trabalho atual
current_directory = os.getcwd()

# Exibe o diretório de trabalho atual
print('Diretório atual:', current_directory)

Diretório atual: c:\Users\Usuario\Downloads\aulas_marcus\modern-ml


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import seaborn as sns
from global_code.util import reduce_mem_usage
from catboost import CatBoostClassifier
import arfs.feature_selection.allrelevant as arfsgroot
import gc
from sklearn.model_selection import KFold, TimeSeriesSplit
import optuna
from sklearn.metrics import log_loss, roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve
import json
import joblib
from global_code.util import reduce_mem_usage, clf_metric_report, compute_and_plot_permutation_importance, plot_pr_calib_curve, plot_dis_probs, plot_shap_values
from sklearn.calibration import CalibratedClassifierCV
from venn_abers import VennAbersCalibrator

# Charts Setup
plt.rcParams.update(**{'figure.dpi': 150})
# color palette can be passed as a list of hex codes
custom_colors = ["#9b59b6", "#3498db", "#95a5a6", "#e74c3c", "#34495e", "#2ecc71"]
# set overall plot style, font size scaling factor, and color palette
sns.set_theme(style="whitegrid", font_scale=1, palette=custom_colors)

c:\Users\Usuario\Downloads\aulas_marcus\modern-ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Lendo a base de dados Parquet
df = pd.read_parquet('./data/lending_club_case_train_dataset.parquet')


# Mapeando o status do empréstimo
df.loc[:, 'default'] = df.loan_status.map({'Fully Paid': 0, 'Charged Off': 1})
df = df.dropna(subset=['default'])
df = df.reset_index(drop=True)
df_original=df

# Exibindo informações sobre o DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1583646 entries, 0 to 1583645
Columns: 142 entries, id to default
dtypes: Int16(54), Int32(15), Int64(15), category(22), datetime64[ns](9), float16(23), float32(4)
memory usage: 922.5 MB


In [5]:
df_default_nulo = df[df['default'].isnull()]

# Exibindo os valores únicos da variável 'loan_status' nesses casos
valores_unicos_loan_status = df_default_nulo['loan_status'].unique()
valores_unicos_loan_status

[], Categories (10, object): ['Charged Off', 'Current', 'Default', 'Does not meet the credit policy. Status:Charg..., ..., 'In Grace Period', 'Issued', 'Late (16-30 days)', 'Late (31-120 days)']

In [6]:
df = reduce_mem_usage(df)

Memory usage of dataframe is 922.53 MB
Memory usage after optimization is: 922.53 MB
Decreased by 0.0%


### ORGANIZANDO FEATURES

In [7]:
df = df.copy()
# Lista de colunas com dados coletados após a concessão do crédito, ou seja, após o processo de aplicação (application)
after_grant_cols= ["loan_amnt","loan_status","pymnt_plan","out_prncp","out_prncp_inv","total_pymnt","total_pymnt_inv","total_rec_prncp","total_rec_int","total_rec_late_fee","recoveries","collection_recovery_fee","last_pymnt_d",
    "last_pymnt_amnt","next_pymnt_d","last_credit_pull_d","last_fico_range_high","last_fico_range_low","hardship_flag","hardship_type","hardship_reason",
    "hardship_status","deferral_term","hardship_amount","hardship_start_date","hardship_end_date","payment_plan_start_date","hardship_length",
    "hardship_dpd","hardship_loan_status","orig_projected_additional_accrued_interest","hardship_payoff_balance_amount","hardship_last_payment_amount","debt_settlement_flag"]

# Removendo as variáveis relacionadas ao plano de dificuldades (covid)
# https://structuredfinance.org/resource-details/helping-consumers-bridge-financial-hardship/
hardship_cols = [
                    'hardship_amount','hardship_start_date','hardship_end_date','payment_plan_start_date',
                    'hardship_length','hardship_dpd','hardship_loan_status',
                    'orig_projected_additional_accrued_interest','hardship_payoff_balance_amount',
                    'hardship_last_payment_amount'
                ]

useless_cols = ["id","url","title","zip_code","addr_state","emp_title"]

date_cols = ["earliest_cr_line","last_pymnt_d","next_pymnt_d","last_credit_pull_d","hardship_start_date","hardship_end_date","payment_plan_start_date"]

# Colunas a serem dropadas
cols_to_drop = after_grant_cols + hardship_cols + useless_cols + date_cols

# feature columns
feature_cols = df.drop(columns=cols_to_drop).columns #+ ['default'] - nao vou dropar o default aqui

# Removendo as colunas do dataset
df = df[feature_cols].copy()

print(' ')
print('Colunas dropadas: ', cols_to_drop)
print('Shape original, ', df.shape)
print('Shape atualizado, ', df.shape)

 
Colunas dropadas:  ['loan_amnt', 'loan_status', 'pymnt_plan', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'debt_settlement_flag', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'id', 'url', 'titl

In [8]:
df['issue_year'] = df['issue_d'].dt.year

# Agrupa por ano e conta o número de registros
registros_por_ano = df.groupby('issue_year').size()

# Exibe o resultado
print(registros_por_ano)

issue_year
2007       251
2008      1562
2009      4716
2010     11536
2011     21721
2012     53367
2013    134814
2014    235616
2015    402819
2016    403032
2017    314212
dtype: int64


### Base Teste x Validação

In [9]:
#Usei apenas 2015 a 2017, visto que houve um boom nas solicitacoes desse periodo
df_train = df[(df['issue_d'] >= '2015-01-01') & (df['issue_d'] < '2017-01-01')] #855.502 registros
df_validation = df[(df['issue_d'] >= '2017-01-01') & (df['issue_d'] < '2017-09-01')] #285.218 registros
df_calibration = df[(df.issue_d >= '2017-09-01')] #158.361 registros

### Feature Selection

In [10]:
target = 'default'

# Input variables and Target dataframes
X_train, y_train= df_train.drop(target, axis=1), df_train.loc[:, target]
X_validation, y_validation= df_validation.drop(target, axis=1), df_validation.loc[:, target]
X_calibration, y_calibration= df_calibration.drop(target, axis=1), df_calibration.loc[:, target]

# Freeing memory
train_df = None
calibration_df = None
validation_df = None
gc.collect()

print('Train Shape: ', X_train.shape, y_train.shape)
print('Validation shape: ', X_validation.shape, y_validation.shape)
print('Calibration shape: ', X_calibration.shape, y_calibration.shape)

Train Shape:  (805851, 101) (805851,)
Validation shape:  (222835, 101) (222835,)
Calibration shape:  (91377, 101) (91377,)


In [11]:
#Retirar variaveis categóricas 
cat_features = df.select_dtypes(include=['object', 'category','datetime']).columns.tolist()
date_features = df.select_dtypes(include=['datetime']).columns.tolist()
print(date_features)

['issue_d', 'sec_app_earliest_cr_line']


In [12]:
# untuned, this is just a set of "reasonable defaults" to get better feature selection
feat_selection_params = {
    'random_strength': 1,
    'learning_rate': 0.02,
    'max_depth': 8,
    'colsample_bylevel': 0.8,
    'subsample': 0.7,
    'random_seed': 42,
    'auto_class_weights': 'Balanced',
    # 'cat_features': cat_features,
    'verbose': False 
}

''''
Iteration: 	1 / 10
Confirmed: 	37
Tentative: 	47
Rejected: 	66
'''

model_for_feat_selection = CatBoostClassifier(**feat_selection_params) 

feat_selector = arfsgroot.Leshy(
    model_for_feat_selection, n_estimators=150, verbose=1, max_iter=10, random_state=55, importance="fastshap",
)

feat_selector.fit(X_train.drop(cat_features, axis=1), y_train)

c:\Users\Usuario\Downloads\aulas_marcus\modern-ml\.venv\Lib\site-packages\arfs\feature_selection\allrelevant.py:321: UserWarning: fasttreeshap is not installed. Fallback to shap.
  warnings.warn("fasttreeshap is not installed. Fallback to shap.")
c:\Users\Usuario\Downloads\aulas_marcus\modern-ml\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
Leshy iteration:  90%|█████████ | 9/10 [34:25<03:49, 229.53s/it]




Leshy finished running using shap var. imp.

Iteration: 	1 / 10
Confirmed: 	47
Tentative: 	17
Rejected: 	26
All relevant predictors selected in 00:34:31.51


c:\Users\Usuario\Downloads\aulas_marcus\modern-ml\.venv\Lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The Leshy or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


Leshy(estimator=<catboost.core.CatBoostClassifier object at 0x000002A6CEE55550>,
      max_iter=10, n_estimators=150,
      random_state=RandomState(MT19937) at 0x2A695A6F940, verbose=1)

In [13]:
selected_features = feat_selector.get_feature_names_out().tolist() + cat_features 

variaveis_para_remover = ['emp_length', 'verification_status_joint', 'revol_util', 'sec_app_earliest_cr_line', 'issue_d']

selected_features = [var for var in selected_features if var not in variaveis_para_remover]
cat_features = [var for var in cat_features if var in selected_features]
# selected_features

### Tuning

In [ ]:
best_params = None

In [ ]:
# Types for tuning hipeparameters: https://www.linkedin.com/posts/timurbikmukhametov_9-practical-tips-for-tuning-gradient-boosting-activity-7285985903984955392-IRHR?utm_source=share&utm_medium=member_desktop
def objective(trial):
    params = {
        'iterations': trial.suggest_categorical('iterations', [50, 100, 150, 200, 300, 500, 750, 1000]),
        'max_depth': trial.suggest_categorical('depth', [4, 6, 8, 10, 12, 14]),
        'colsample_bylevel': trial.suggest_categorical('colsample_bylevel', [0.5, 0.7, 0.8, 0.9, 1.0]),
        'subsample': trial.suggest_categorical('subsample', [0.5, 0.7, 0.8, 1.0]),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True),
        'auto_class_weights': trial.suggest_categorical('auto_class_weights', ['Balanced', 'SqrtBalanced']),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bernoulli']),
        'cat_features': cat_features,
        'verbose': 0
    }

    model = CatBoostClassifier(**params, eval_metric='PRAUC:use_weights=false')
    tscv = TimeSeriesSplit(n_splits=4)
    avg_precision_scores = []

    for train_index, val_index in tscv.split(X_train[selected_features]):
        X_train_fold, X_val_fold = X_train[selected_features].iloc[train_index], X_train[selected_features].iloc[val_index]
        y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

        model.fit(X_train_fold, y_train_fold, eval_set=(X_val_fold, y_val_fold), early_stopping_rounds=200)
        #model.fit(X_train_fold, y_train_fold)
        y_pred_fold = model.predict_proba(X_val_fold)[:, 1]
        avg_precision_scores.append(average_precision_score(y_val_fold, y_pred_fold))

    return np.mean(avg_precision_scores)

study = optuna.create_study(direction='maximize')

# Optimize for 1 hour 
# print('Tuning the model...')
#study.optimize(objective, timeout=3600)

# Optimize for 1 minute
print('Tuning the model...')
study.optimize(objective, timeout=60*60*15, n_trials=10)

best_params = study.best_params
print(f'Best parameters: {best_params}')

[I 2025-02-06 00:28:02,220] A new study created in memory with name: no-name-a93c7294-3550-48a1-bc8f-eeb361e068f0


Tuning the model...


[I 2025-02-06 00:29:17,772] Trial 0 finished with value: 0.31663542834220965 and parameters: {'iterations': 50, 'depth': 4, 'colsample_bylevel': 0.7, 'subsample': 0.8, 'learning_rate': 0.0011603480329971435, 'auto_class_weights': 'Balanced', 'bootstrap_type': 'Bernoulli'}. Best is trial 0 with value: 0.31663542834220965.
[I 2025-02-06 08:45:12,699] Trial 1 finished with value: 0.37472834062482696 and parameters: {'iterations': 500, 'depth': 10, 'colsample_bylevel': 0.5, 'subsample': 0.5, 'learning_rate': 0.00017755260134782663, 'auto_class_weights': 'Balanced', 'bootstrap_type': 'Bernoulli'}. Best is trial 1 with value: 0.37472834062482696.
[I 2025-02-06 08:47:39,493] Trial 2 finished with value: 0.37314706740214804 and parameters: {'iterations': 50, 'depth': 10, 'colsample_bylevel': 0.5, 'subsample': 0.7, 'learning_rate': 0.0006743315368858713, 'auto_class_weights': 'Balanced', 'bootstrap_type': 'Bernoulli'}. Best is trial 1 with value: 0.37472834062482696.
[W 2025-02-06 19:55:42,910]

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params
print(f'Best parameters: {best_params}')

Best parameters: {'iterations': 500, 'depth': 10, 'colsample_bylevel': 0.5, 'subsample': 0.5, 'learning_rate': 0.00017755260134782663, 'auto_class_weights': 'Balanced', 'bootstrap_type': 'Bernoulli'}


In [ ]:
# Save the best parameters to a JSON file
print('Saving the best parameters to a JSON file...\n ', json.dumps(best_params, indent=4))
best_params_path = './case/model/best_params.json'
with open(best_params_path, 'w') as f:
    json.dump(best_params, f, indent=4)

Saving the best parameters to a JSON file...
  {
    "iterations": 500,
    "depth": 10,
    "colsample_bylevel": 0.5,
    "subsample": 0.5,
    "learning_rate": 0.00017755260134782663,
    "auto_class_weights": "Balanced",
    "bootstrap_type": "Bernoulli"
}


In [14]:
#ajustar o best_params
best_params = None
if not best_params: 
    best_params = {
    "iterations": 500,
    "depth": 10,
    "colsample_bylevel": 0.5,
    "subsample": 0.5,
    "learning_rate": 0.00017755260134782663,
    "auto_class_weights": "Balanced",
    "bootstrap_type": "Bernoulli"
    }
    #best_params = {'iterations': 750, 'depth': 8, 'colsample_bylevel': 0.8, 'subsample': 0.5, 'learning_rate': 0.01, 'auto_class_weights': 'SqrtBalanced'}

model_tunned = CatBoostClassifier(**best_params, eval_metric='PRAUC:use_weights=false', cat_features=cat_features, random_state=125)
model_tunned.fit(X_train[selected_features], y_train, eval_set=(X_validation[selected_features], y_validation))

y_pred_tunned = model_tunned.predict_proba(X_validation[selected_features])[:, 1]

0:	learn: 0.3694566	test: 0.3557464	best: 0.3557464 (0)	total: 2.11s	remaining: 17m 35s
1:	learn: 0.3706821	test: 0.3563109	best: 0.3563109 (1)	total: 4.17s	remaining: 17m 17s
2:	learn: 0.3782147	test: 0.3627107	best: 0.3627107 (2)	total: 5.83s	remaining: 16m 5s
3:	learn: 0.3799773	test: 0.3643366	best: 0.3643366 (3)	total: 7.78s	remaining: 16m 4s
4:	learn: 0.3803065	test: 0.3648340	best: 0.3648340 (4)	total: 9.84s	remaining: 16m 13s
5:	learn: 0.3809527	test: 0.3653163	best: 0.3653163 (5)	total: 12.1s	remaining: 16m 36s
6:	learn: 0.3817065	test: 0.3661155	best: 0.3661155 (6)	total: 13.8s	remaining: 16m 13s
7:	learn: 0.3815744	test: 0.3661845	best: 0.3661845 (7)	total: 15.8s	remaining: 16m 9s
8:	learn: 0.3818831	test: 0.3664792	best: 0.3664792 (8)	total: 17.9s	remaining: 16m 15s
9:	learn: 0.3825575	test: 0.3667300	best: 0.3667300 (9)	total: 19.8s	remaining: 16m 8s
10:	learn: 0.3827891	test: 0.3667805	best: 0.3667805 (10)	total: 21.9s	remaining: 16m 11s
11:	learn: 0.3826198	test: 0.36692

In [15]:
# Model metrics report
clf_metric_report(y_pred_tunned, y_validation)

Evaluating the model...
ROC AUC: 0.7071912435194091
Brier Score: 0.24977984353923982
Average Precision: 0.36725141035789205
Log Loss: 0.6927068671306584


In [ ]:
# Save the model trained with selected features
tunned_model_path = './case/model/tunned_model.joblib'
joblib.dump(model_tunned, tunned_model_path)

print(f"Baseline model saved to: {tunned_model_path}")

Baseline model saved to: ./case/model/tunned_model.joblib


In [16]:
tunned_model=model_tunned
champion_model=model_tunned

In [17]:
# Platt scaling (sigmoid)
print('Fitting platt scaling calibration...')
calibrated_model_sigmoid = CalibratedClassifierCV(champion_model, method='sigmoid')
calibrated_model_sigmoid.fit(X_calibration[selected_features], y_calibration)
y_pred_sigmoid = calibrated_model_sigmoid.predict_proba(X_validation[selected_features])[:, 1]

Fitting platt scaling calibration...
0:	learn: 0.4101793	total: 307ms	remaining: 2m 33s
1:	learn: 0.4364254	total: 650ms	remaining: 2m 41s
2:	learn: 0.4384610	total: 1.04s	remaining: 2m 53s
3:	learn: 0.4422134	total: 1.4s	remaining: 2m 53s
4:	learn: 0.4420685	total: 1.78s	remaining: 2m 56s
5:	learn: 0.4425295	total: 2.11s	remaining: 2m 53s
6:	learn: 0.4422874	total: 2.51s	remaining: 2m 56s
7:	learn: 0.4437948	total: 2.86s	remaining: 2m 56s
8:	learn: 0.4422011	total: 3.17s	remaining: 2m 53s
9:	learn: 0.4422690	total: 3.53s	remaining: 2m 52s
10:	learn: 0.4465966	total: 3.89s	remaining: 2m 52s
11:	learn: 0.4463198	total: 4.24s	remaining: 2m 52s
12:	learn: 0.4463941	total: 4.57s	remaining: 2m 51s
13:	learn: 0.4471376	total: 4.93s	remaining: 2m 51s
14:	learn: 0.4476379	total: 5.27s	remaining: 2m 50s
15:	learn: 0.4484250	total: 5.64s	remaining: 2m 50s
16:	learn: 0.4489527	total: 6.01s	remaining: 2m 50s
17:	learn: 0.4484862	total: 6.38s	remaining: 2m 51s
18:	learn: 0.4500344	total: 6.75s	rema

In [18]:
# Isotonic regression
print('Fitting isotonic regression calibration...')
calibrated_model_isotonic = CalibratedClassifierCV(champion_model, method='isotonic')
calibrated_model_isotonic.fit(X_calibration[selected_features], y_calibration)
y_pred_isotonic = calibrated_model_isotonic.predict_proba(X_validation[selected_features])[:, 1]

Fitting isotonic regression calibration...
0:	learn: 0.4101793	total: 394ms	remaining: 3m 16s
1:	learn: 0.4364254	total: 814ms	remaining: 3m 22s
2:	learn: 0.4384610	total: 1.2s	remaining: 3m 18s
3:	learn: 0.4422134	total: 1.55s	remaining: 3m 12s
4:	learn: 0.4420685	total: 1.95s	remaining: 3m 13s
5:	learn: 0.4425295	total: 2.31s	remaining: 3m 10s
6:	learn: 0.4422874	total: 2.72s	remaining: 3m 11s
7:	learn: 0.4437948	total: 3.08s	remaining: 3m 9s
8:	learn: 0.4422011	total: 3.32s	remaining: 3m 1s
9:	learn: 0.4422690	total: 3.62s	remaining: 2m 57s
10:	learn: 0.4465966	total: 3.93s	remaining: 2m 54s
11:	learn: 0.4463198	total: 4.23s	remaining: 2m 51s
12:	learn: 0.4463941	total: 4.53s	remaining: 2m 49s
13:	learn: 0.4471376	total: 4.84s	remaining: 2m 47s
14:	learn: 0.4476379	total: 5.14s	remaining: 2m 46s
15:	learn: 0.4484250	total: 5.46s	remaining: 2m 45s
16:	learn: 0.4489527	total: 5.8s	remaining: 2m 44s
17:	learn: 0.4484862	total: 6.11s	remaining: 2m 43s
18:	learn: 0.4500344	total: 6.42s	r

### Venn-abers

In [19]:
p_cal = tunned_model.predict_proba(X_calibration[selected_features])
p_test = tunned_model.predict_proba(X_validation[selected_features])

va = VennAbersCalibrator()
#Ajuste na base de validação
va_prefit_prob = va.predict_proba(p_cal=p_cal, y_cal=y_calibration.values, p_test=p_test)
y_pred_va = va_prefit_prob[:, 1]

In [20]:
# Compute metrics for Platt scaling and isotonic regression
print("Platt Scaling (Sigmoid) Metrics:")
clf_metric_report(y_pred_sigmoid, y_validation)

print("\nIsotonic Regression Metrics:")
clf_metric_report(y_pred_isotonic, y_validation)

# Compute metrics for Venn-Abers calibration
print("\nVenn-Abers Calibration Metrics:")
clf_metric_report(y_pred_va, y_validation)

Platt Scaling (Sigmoid) Metrics:
Evaluating the model...
ROC AUC: 0.7064471539144447
Brier Score: 0.14640721452709726
Average Precision: 0.365961604822504
Log Loss: 0.4582014410667873

Isotonic Regression Metrics:
Evaluating the model...
ROC AUC: 0.7064401119612598
Brier Score: 0.1462431509034097
Average Precision: 0.3650581968759721
Log Loss: 0.45790782784988493

Venn-Abers Calibration Metrics:
Evaluating the model...
ROC AUC: 0.7070610018883712
Brier Score: 0.1459526460321535
Average Precision: 0.3621745044509709
Log Loss: 0.45761209211812337


In [21]:
df_test = pd.read_parquet('./data/lending_club_case_case_test_dataset.parquet')

In [22]:
# Mapeando o status do empréstimo
df_test.loc[:, 'default'] = df_test.loan_status.map({'Fully Paid': 0, 'Charged Off': 1})
df_test = df_test.dropna(subset=['default'])
df_test = df_test.reset_index(drop=True)

In [23]:
#Ajuste na base de teste
p_cal = tunned_model.predict_proba(X_calibration[selected_features])
p_test = tunned_model.predict_proba(df_test[selected_features])

va = VennAbersCalibrator()

va_prefit_prob = va.predict_proba(p_cal=p_cal, y_cal=y_calibration.values, p_test=p_test)
y_pred_va = va_prefit_prob[:, 1]

In [ ]:
submission = pd.DataFrame({'id': df_test['id'], 'default_probability': y_pred_va})
submission.to_csv('matheus_braga_submission.csv', index=False)

print('Matheus Braga Submission saved to submission.csv')

Matheus Braga Submission saved to submission.csv


### Credit Policy

In [24]:
df_cp= df_original[(df_original.issue_d >= '2017-09-01')] #158.361 registros
# Total amount owed on the loan (principal + interest)
df_cp['principal'] = df_cp['loan_amnt'] * (1 + df_cp["int_rate"])
df_cp['gain'] = df_cp['principal']-df_cp['loan_amnt'] 

# Exposure at Default (EAD): portion of the total owed that remains unpaid
df_cp['ead'] = df_cp['principal'] - df_cp.total_pymnt
df_cp['ead_ratio'] = df_cp['ead'] / df_cp['principal']

# Loss Given Default (LGD): 1 - recoveries ratio
df_cp['lgd_ratio'] = 1 - (df_cp['recoveries'] / df_cp['ead'])

# Final Losses
df_cp['losses'] = (
    df_cp['principal'] 
    - df_cp.total_pymnt 
    + df_cp['recoveries']
)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_14272\740717634.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cp['principal'] = df_cp['loan_amnt'] * (1 + df_cp["int_rate"])
C:\Users\Usuario\AppData\Local\Temp\ipykernel_14272\740717634.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cp['gain'] = df_cp['principal']-df_cp['loan_amnt']
C:\Users\Usuario\AppData\Local\Temp\ipykernel_14272\740717634.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

### Adiciona as probabiblidade à base de calibração

In [25]:
p_cal = tunned_model.predict_proba(X_calibration[selected_features])
p_test = tunned_model.predict_proba(X_calibration[selected_features])

va = VennAbersCalibrator()
#Ajuste na base de validação
va_prefit_prob = va.predict_proba(p_cal=p_cal, y_cal=y_calibration.values, p_test=p_test)
y_pred_va = va_prefit_prob[:, 1]

In [38]:
df_calibration.loc[:, 'losses'] = df_cp['losses']
df_calibration.loc[:, 'gains'] = df_cp['gain']
df_calibration.loc[:,'loan_amnt']= df_cp['loan_amnt']
df_calibration.loc[:, 'default_proba'] = y_pred_va
df_calibration.loc[:, 'term_in_years'] = df_calibration.loc[:, 'term'] // 12
df_calibration.loc[:, 'roi_per_year'] = ((df_calibration.loc[:, 'gains'] - df_calibration.loc[:, 'losses']) / df_calibration.loc[:, 'loan_amnt'])/df_calibration.loc[:, 'term_in_years']

### Política final

In [41]:
# Classificar os registros por 'default_proba'
# Classificar os registros por 'default_proba'
df_calibration = df_calibration.sort_values(by='default_proba')

# Dividir em 5 ratings
df_calibration['rating'] = pd.qcut(df_calibration['default_proba'], 5, labels=['A', 'B', 'C', 'D', 'E'], duplicates='drop')

# Agrupar por rating e calcular as métricas, incluindo default_ratio
rating_summary = df_calibration.groupby('rating', observed=False).apply(
    lambda x: pd.Series({
        'default_proba_mean': x['default_proba'].mean(),
        'default_ratio': x['default'].sum() / x['default'].count(),
        'record_count': x.shape[0],
        'roi_per_year_mean': (x['roi_per_year'] * x['loan_amnt']).sum() / x['loan_amnt'].sum()
    })
).reset_index()

# Visualizar a tabela
print(rating_summary)

  rating  default_proba_mean  default_ratio  record_count  roi_per_year_mean
0      A            0.082148       0.081451       18465.0           0.005261
1      B            0.158655       0.158244       18086.0          -0.002597
2      C            0.230452       0.230286       21191.0          -0.013578
3      D            0.308151       0.307792       15387.0          -0.023963
4      E            0.441500       0.441473       18248.0          -0.040144


C:\Users\Usuario\AppData\Local\Temp\ipykernel_14272\2666704.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rating_summary = df_calibration.groupby('rating', observed=False).apply(
